# Qwen 50-invoice rerun on the shared OCR holdout

Benchmark Qwen3-0.6B base, the 0.6B LoRA adapter, and Qwen3-1.7B base on the same 50 cached OCR records. This run reuses the completed adapter and does not retrain. Results are saved under `/kaggle/working/invoice-llm-rerun-20261112/qwen`.


In [ ]:
from pathlib import Path
import csv, json, shutil, zipfile, random, os, time, gc, logging, traceback, unicodedata, statistics
import numpy as np
from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
for secret_name in ("GROQ_API_KEY", "GEMINI_API"):
    try:
        secret_value = secret_client.get_secret(secret_name)
    except Exception as error:
        raise RuntimeError(f"Cannot access Kaggle secret {secret_name}: {type(error).__name__}") from None
    if not secret_value:
        raise RuntimeError(f"Missing Kaggle secret {secret_name}; attach it to this benchmark notebook.")
    os.environ[secret_name] = secret_value
del secret_client, secret_value
print("Both provider secrets are available.")

input_root = Path('/kaggle/input')
out=Path('/kaggle/working/invoice-llm-rerun-20261112/api')
out.mkdir(parents=True,exist_ok=True)
manifests = list(input_root.rglob('manifest.csv'))
if len(manifests) == 1:
    data_root = manifests[0].parent
else:
    archives = list(input_root.rglob('invoice-synthetic-1000-sft.zip'))
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one manifest.csv or pilot ZIP; manifests={manifests}; archives={archives}; input_root={list(input_root.iterdir())}')
    data_parent = Path('/kaggle/working/invoice-synthetic-1000')
    shutil.rmtree(data_parent, ignore_errors=True)
    data_parent.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(data_parent)
    data_root = data_parent / 'dataset'
manifest = list(csv.DictReader((data_root / 'manifest.csv').open(encoding='utf-8-sig')))
ground_truth = json.loads((data_root / 'ground_truth.json').read_text(encoding='utf-8'))
assert len(manifest) == len(ground_truth) == 1000
assert {s: sum(r['split'] == s for r in manifest) for s in ('train','validation','test')} == {'train':900,'validation':50,'test':50}
by_file = {row['file']: (row, ground_truth[i]) for i, row in enumerate(manifest)}
print('Verified heldout dataset manifest.')


## 2. Install only the extraction-model dependencies

The benchmark reuses the previously saved Docling/VietOCR output for the exact 50 held-out invoices, so this run does not initialize the OCR stack.


In [ ]:
import subprocess, sys
repo = Path('/tmp/enterprise-knowledge-ai-agent')
shutil.rmtree(repo, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/Lenhan231/enterprise-knowledge-ai-agent.git',str(repo)],check=True)
source_commit = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
processor_sources=list(input_root.rglob('invoice_text_processor.py'))
if len(processor_sources)!=1: raise FileNotFoundError(f'Expected one bundled invoice_text_processor.py; found {processor_sources}')
shutil.copy2(processor_sources[0],repo/'src/document_processing/invoice_text_processor.py')
sys.path.insert(0,str(repo/'src'))
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','openai==2.41.0'],check=True)


## 3. Build production-shaped supervised examples

Each invoice contributes a header task and a table task. The serialized evidence matches the production prompts and schema. Column headers and cell values are built from the labeled invoice; this training setup does not include OCR errors.

In [ ]:
# Not needed for provider API inference.


## 4. Fine-tune Qwen3-0.6B with LoRA

Use a frozen 0.6B base with attention and MLP LoRA adapters. Prompt tokens are masked from loss; only the target JSON is trained.

In [ ]:
# Not needed for provider API inference.


In [ ]:
if RUN_TRAINING:
    from torch.nn.utils.rnn import pad_sequence

    class CausalOnlyCollator:
        def __call__(self,features):
            input_ids=[torch.tensor(x['input_ids'],dtype=torch.long) for x in features]
            labels=[torch.tensor(x['labels'],dtype=torch.long) for x in features]
            return {
                'input_ids':pad_sequence(input_ids,batch_first=True,padding_value=tokenizer.pad_token_id),
                'labels':pad_sequence(labels,batch_first=True,padding_value=-100),
                'attention_mask':pad_sequence([torch.ones_like(x) for x in input_ids],batch_first=True,padding_value=0),
            }

    base_model=AutoModelForCausalLM.from_pretrained(MODEL_06,revision=REV_06,torch_dtype=torch.float16)
    base_model.config.use_cache=False
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()
    base_model=get_peft_model(base_model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    base_model.print_trainable_parameters()
    # Keep the trainable model in memory for the Trainer cell below.
else:
    print('No training model loaded; GPU memory available for OCR.')


In [ ]:
# Not needed for provider API inference.


## 5. Load cached OCR for the 50 held-out PDFs

All three models receive the same previously saved Docling/VietOCR elements and table cells from these exact held-out invoices.


In [ ]:
cache_files=list(input_root.rglob('holdout-ocr.json'))
if len(cache_files)!=1: raise FileNotFoundError(f'Expected one cached holdout-ocr.json; found {cache_files}')
ocr_cache=json.loads(cache_files[0].read_text(encoding='utf-8'))
expected_test={r['file']:by_file[r['file']][1] for r in manifest if r['split']=='test'}
if len(ocr_cache)!=50 or {r['file'] for r in ocr_cache}!=set(expected_test):
    raise ValueError('Cached OCR records do not match the 50-file held-out cohort')
for record in ocr_cache:
    if record['expected']!=expected_test[record['file']]:
        raise ValueError(f"Cached label mismatch for {record['file']}")
    record['ocr_status']='success'
    record['ocr_seconds']=None
print('Loaded OCR cache:',cache_files[0])
print('50/50 exact holdout files; elements:',sum(len(r['elements']) for r in ocr_cache),'tables:',sum(len(r['tables']) for r in ocr_cache))
print('OCR pipeline: Docling 2.129.0 + fine-tuned VietOCR, cached from the established 50-invoice pilot.')


## 6. Benchmark base 0.6B, fine-tuned 0.6B, and base 1.7B

Inference uses the repository production job builder and merge/parser functions. Greedy decoding, same prompts, same OCR cache, same schema, and same 50 holdout files.

In [ ]:
print('Qwen inference skipped in this API-only version; see the Qwen benchmark version.')


## 7. Score and save the comparison

Exact match is normalized for Unicode whitespace/case; numeric fields compare digits only, matching the previous pilot metric. Item rows are matched by position. Both macro-per-invoice and pooled row accuracy are reported.

In [ ]:
print('Qwen scoring skipped in this API-only version.')


## 8. Export the adapter and benchmark artifacts

The adapter is saved separately from the base model. Metrics and per-invoice outputs are preserved for review.

In [ ]:
import httpx
import openai
import document_processing.invoice_text_processor as invoice_module
from document_processing.invoice_text_processor import InvoiceTextProcessor

groq_last_request = [0.0]
def pace_groq(request):
    if "api.groq.com" not in str(request.url):
        return
    delay = 2.1 - (time.monotonic() - groq_last_request[0])
    if delay > 0:
        time.sleep(delay)
    groq_last_request[0] = time.monotonic()

BaseOpenAI = openai.OpenAI
class PacedOpenAI(BaseOpenAI):
    def __init__(self, *args, **kwargs):
        kwargs["http_client"] = httpx.Client(event_hooks={"request": [pace_groq]})
        super().__init__(*args, **kwargs)
openai.OpenAI = PacedOpenAI

original_urlopen = invoice_module.urlopen
gemini_last_request = [0.0]
def pace_gemini(request, *args, **kwargs):
    url = request.full_url if hasattr(request, "full_url") else str(request)
    if "generativelanguage.googleapis.com" in url:
        delay = 4.1 - (time.monotonic() - gemini_last_request[0])
        if delay > 0:
            time.sleep(delay)
        gemini_last_request[0] = time.monotonic()
    return original_urlopen(request, *args, **kwargs)
invoice_module.urlopen = pace_gemini

api_out = out
for provider, config in (
    ("groq", {"invoice_provider": "groq", "groq_invoice_model": "openai/gpt-oss-120b"}),
    ("gemini", {"invoice_provider": "gemini", "gemini_invoice_model": "gemini-3.1-flash-lite"}),
):
    target = api_out / f"{provider}-results.json"
    data = json.loads(target.read_text(encoding="utf-8")) if target.exists() else {
        "metadata": {
            "provider": provider,
            "model": config.get("groq_invoice_model") or config.get("gemini_invoice_model"),
            "sample_count": len(ocr_cache),
            "shared_ocr_cache": True,
            "source_commit": source_commit,
            "ocr_pipeline": "Docling 2.129.0 + fine-tuned VietOCR cached holdout",
        },
        "results": [],
    }
    completed = {row["file"] for row in data["results"] if row.get("status") == "success"}
    processor = InvoiceTextProcessor(config)
    for index, record in enumerate(ocr_cache, 1):
        if record["file"] in completed:
            continue
        row = {
            "file": record["file"],
            "quality": record["quality"],
            "expected": record["expected"],
            "status": "failed",
        }
        started = time.perf_counter()
        try:
            extraction = processor.process(record.get("elements") or [], record.get("tables") or [])
            row.update({
                "status": "success" if extraction.get("status") == "review_required" else extraction.get("status", "failed"),
                "schema_valid": extraction.get("schema_version") == "invoice.v1" and extraction.get("fields") is not None,
                "fields": extraction.get("fields") or {},
                "model": extraction.get("model"),
                "revision": extraction.get("revision"),
                "metrics": extraction.get("metrics") or {},
                "parse_error": extraction.get("parse_error"),
                "field_conflicts": extraction.get("field_conflicts") or {},
                "raw_response": extraction.get("raw_response"),
            })
        except Exception as error:
            row["error"] = f"{type(error).__name__}: {error}"
        row["elapsed_seconds"] = round(time.perf_counter() - started, 2)
        data["results"].append(row)
        temp = target.with_suffix(".tmp")
        temp.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        temp.replace(target)
        print(f"{provider.upper()} {index}/{len(ocr_cache)} {record['file']} {row['elapsed_seconds']}s {row['status']}", flush=True)
print("API benchmark outputs:", api_out)

